
# Do LLMs know the difference between a pet chicken and a roast chicken?

## Word sense disambiguation in computational models and humans


In human language, words do not always have a fixed meaning. The most striking example is homonymous words: words that have the same form, but very different meanings. For instance, the word "bank", which has a different meaning in the context "I went to the bank to get some money" and "At the river bank, I met my old friend". Polysemous words are words that have different -- yet related -- meanings: for example, "chicken" is the same 'entity' in "My pet chicken is lovely" and "I am having roast chicken for dinner", but has very different meanings in these two contexts. In general, context can modulate almost any word's meaning. This poses a challenge in computational linguistics, as we need to find a way to differentiate among different meanings like humans do. Much research, resources, and models have been put forward to help with this challenge.

In this assignment, you are going to focus on [Trott and Bergen's (2021)](https://aclanthology.org/2021.acl-long.550/) RAW-C dataset: you are going to conduct a number of explorations with this dataset and partially replicate their research by the end of the assignment. In short, the authors explore how good LLMs are at capturing same/different meanings of words across contexts by comparing it to human judgements. To better understand the idea and the research, start by reading the paper.

This assignment entails a series of (interconnected) tasks (altogether worth 95 points):

* **Task 1**. Compute contextual word embeddings at different layers from Trott & Bergen's dataset. Here, each word is found in 4 sentences: 2 with one meaning, 2 with another meaning.
* **Task 2**. Compute sense embeddings for words in Trott & Bergen's dataset using WordNet, so you have an embedding for each definition of the word.
* **Task 3**. Compute the similarity between the contextual word embeddings of the homonyms at different layers and their sense embeddings; explore the relationship between homonyms and dominant senses quantitatively and qualitatively
* **Task 4**. Replicate part of Trott & Bergen's work by computing similarities across sentences with same/different meanings at the different layers and correlate with human similarities; visualise the results and reflect on them

In order to better understand the assignment, we recommend going through it all before starting so that it is clear how each part is connected to the next (which will help you make decisions about data structures, for instance).

# Task 1: Compute contextual word embeddings for homonyms [20 points]

## Task 1.1: read, explore and extract the necessary data [5 points]

First, you will have to (fork and) clone the github repository that stores the data you'll need. This can be found here: https://github.com/sashakenjeeva/raw-c . The repo also includes a README with a description of the original files in the repository, as well as some notes relevant for this assignment specifically.

In [1]:
#your code here (you can use as many cells as necessary/you prefer)

Make sure you mount the drive now so that you have access to the folder (think about setting the working directory in a way that is convenient).

In [2]:
# mount the drive here

Now, you will have to read the data and organise it in a structure that works for the next parts of the assignment.

Read and explore the dataframe to see its structure (print part of it). What we need from it are the homonyms (in the form that they appear in the sentence -- the lexeme -- and in their regular form -- the lemma) and their corresponding sentences with different meanings (M1_a and M1_b have same meaning; M2_a, M2_b have same meaning). We only will need the stimuli that are in the final RAW-C dataset, as this is what we'll replicate at the end.

You can decide which data structure to use, but make sure that all these pieces of information are there (the word, the string, the meaning id, and the corresponding sentences) and easy to retrieve. Show your data at the end, as well as how many stimuli you end up with.

In [3]:
import pandas as pd

normed_critical = pd.read_csv("./raw-c/data/processed/normed_critical.csv")
rawc_with_dom = pd.read_csv("./raw-c/data/processed/raw-c_with_dominance.csv")
rawc = pd.read_csv("./raw-c/data/processed/raw-c.csv")
stims_with_nlm_dist = pd.read_csv("./raw-c/data/processed/stims_with_nlm_distances.csv")

In [24]:
rawc.head()

,word,sentence1,sentence2,same,ambiguity_type,disambiguating_word1,disambiguating_word2,version,Class,mean_relatedness,...,count,sd_relatedness,distance_bert,distance_elmo,se_relatedness,v1,v2,string,word_norm,string_norm
0,act,It was a desperate act.,It was a magic act.,False,Polysemy,desperate,magic,M1_a_M2_a,N,2.181818,...,11,1.328020,0.204110,0.034093,0.400413,M1_a,M2_a,act,act,act
1,act,It was a desperate act.,It was a comedic act.,False,Polysemy,desperate,comedic,M1_a_M2_b,N,2.000000,...,7,1.290994,0.215616,0.045927,0.487950,M1_a,M2_b,act,act,act
2,act,It was a humane act.,It was a magic act.,False,Polysemy,humane,magic,M1_b_M2_a,N,2.818182,...,11,0.981650,0.191488,0.042351,0.295979,M1_b,M2_a,act,act,act
3,act,It was a humane act.,It was a comedic act.,False,Polysemy,humane,comedic,M1_b_M2_b,N,2.809524,...,21,0.928388,0.225272,0.057707,0.202591,M1_b,M2_b,act,act,act
4,act,It was a desperate act.,It was a humane act.,True,Polysemy,desperate,humane,M1_a_M1_b,N,3.900000,...,10,0.316228,0.167990,0.041440,0.100000,M1_a,M1_b,act,act,act


## Task 1.2: Compute the contextualised word embeddings [15 points]


Now that you have the homonyms and their corresponding sentences, we will need to compute word embeddings for each of them. For this we will use the BERT base model, in its uncased version.

That is, for each homonym, you will have to compute four embeddings: one for the homonym in M1_a, one in M1_b, one in M2_a, one in M2_b. However, we also want to look into different layers of the BERT model to see which one captures the homonym's meaning best: you want to calculate embeddings at the static layer and at layers 4, 8, 12.

We will use the package psycho-embeddings (you will use it in class), which allows us to specify which target words we want to obtain the embeddings of, in which sentences, and at which layers, among other things. Make sure to read the documentation of the package so that you know the meaning of the arguments and which ones will come useful to you.

First of all, install the psycho-embeddings package below.

In [5]:
# install the psycho-embeddings package here

Now, import the relevant module/function from psycho-embeddings and load the required BERT model.

In [ ]:
import torch
from psycho_embeddings import ContextualizedEmbedder

device = "cuda" if torch.cuda.is_available() else "cpu"

# init the embedder
embedder = ContextualizedEmbedder(
    model_name="bert-base-uncased", # from the task
    max_length=128, # tokens in sentence
    device=device,
)

print(f"Embedder ready on {device}")

d:\computational_linguistics_assignment2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
loading configuration file config.json from cache at C:\Users\maria\.cache\huggingface\hub\models--bert-base-uncased\snapshots\86b5e0934494bd15c9632b12f734a8a67f723594\config.json
Model config BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers"

Embedder ready on cpu


Now, test that everything works correctly by computing an embedding for the word "assignment" in the sentence "I am having so much fun with this assignment!", at static layer and layers 4, 8 and 12 (hint: think of tokenisation and how the embedder deals with that).

In [ ]:
import numpy as np

word = "assignment"
sentence = "I am having so much fun with this assignment!"
layers = [4, 8, 12]

embeddings = embedder.embed(
    words=[word], # words for which it computes the embeddings
    target_texts=[sentence], # sentences providing the context in which the target words are embedded
    layers_id=layers, # layers at which you save vectors
    batch_size=1, # one sentence at a time
    show_progress=False, 
    averaging=True, # to compute an average vector from sub-word units
    return_static=True, # also returns static embedding
)

print("Returned layers:", sorted(embeddings.keys()))

for layer_id in [-1, 4, 8, 12]:
    vec = embeddings[layer_id][0] # vector embedding at each layer ([0] because the list only has the word "assignment")
    print(f"Layer {layer_id}: shape={vec.shape}, norm={np.linalg.norm(vec):.4f}")
    print("  first 8 values:", np.round(vec[:8], 4))

Text tokenization: 100%|██████████| 1/1 [00:00<00:00, 124.25 examples/s]
d:\computational_linguistics_assignment2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Returned layers: [-1, 4, 8, 12]
Layer -1: shape=(768,), norm=1.2572
  first 8 values: [-0.0094 -0.0577 -0.0616 -0.0536 -0.0078 -0.112  -0.0544  0.0357]
Layer 4: shape=(768,), norm=20.9498
  first 8 values: [ 1.9968 -0.591  -0.1317 -0.5903  0.7785 -2.0171  0.6368  0.7959]
Layer 8: shape=(768,), norm=19.8974
  first 8 values: [ 1.0264 -0.6431 -0.3829 -0.276   0.2697 -1.668   0.7257  0.4577]
Layer 12: shape=(768,), norm=13.0209
  first 8 values: [ 0.3949 -0.3286 -0.1683  0.0146  0.0804 -0.8447  0.4655  0.8319]


The next step is to calculate embeddings for the homonyms and their sentences that we got from the RAW-C dataset.

Make sure that your final output includes the word, the meaning id (M1_a, etc), the corresponding sentence and the embeddings at static layer and layers 4, 8, 12. You should maximally optimise this process by calculating in batches (again, check psycho-embeddings documentation), but keep in mind this might still take a while. First test your pipeline with a small number of inputs, and only run the full scale embedding extraction once you're positive the code works as expected.

When done, save the output in [pickle](https://docs.python.org/3/library/pickle.html) format (this is similar to json, but it can also handle np.arrays), so that you can easily load it later when needed and do not have to run it again. After pickle dumping (that's the word for saving it in pickle format), print it so that you are sure everything was saved correctly.

Then, check that your final data includes everything that you need by checking the entry "bank" and print the data pertaining to "bank".

In [ ]:
from pathlib import Path
import pickle
import pandas as pd
import numpy as np

# Rebuild final stimulus set from stimuli.csv + final raw-c.csv pairs
stimuli = pd.read_csv("./raw-c/data/stims/stimuli.csv") 

# lowercase and strip everything because BERT is uncased + match across datasets
stimuli["word_norm"] = stimuli["Word"].str.lower().str.strip()
stimuli["string_norm"] = stimuli["String"].str.lower().str.strip()
rawc["word_norm"] = rawc["word"].str.lower().str.strip()
rawc["string_norm"] = rawc["string"].str.lower().str.strip()

final_pairs = rawc[["word_norm", "string_norm"]].drop_duplicates()

# ADDED THIS (get ambiguity type from rawc)
ambiguity_map = rawc[["word_norm", "string_norm", "ambiguity_type"]].drop_duplicates()

# keep only relevant columns + ambiguity_type
stimuli_final = (
    stimuli.merge(final_pairs, on=["word_norm", "string_norm"], how="inner")
    .merge(ambiguity_map, on=["word_norm", "string_norm"], how="left")  # ← added
    .rename(columns={"Word": "word", "String": "string"})
    [["word", "string", "M1_a", "M1_b", "M2_a", "M2_b", "ambiguity_type"]]
    .sort_values(["word", "string"])
    .reset_index(drop=True)
)

# convert to long format
stimuli_long = (
    stimuli_final
    .melt(
        id_vars=["word", "string", "ambiguity_type"],
        value_vars=["M1_a", "M1_b", "M2_a", "M2_b"],
        var_name="meaning_id",
        value_name="sentence",
    )
    .sort_values(["word", "string", "meaning_id"])
    .reset_index(drop=True)
)

print(f"Final stimuli: {len(stimuli_final)}")
print(f"Sentence entries to embed (4 per stimulus): {len(stimuli_long)}")

layers = [4, 8, 12]
all_layer_ids = [-1, 4, 8, 12]

# Extract contextualized embeddings from BERT for each (word, sentence)
def extract_embeddings(df, batch_size=32):
    rows = df.reset_index(drop=True)
    words = rows["string"].tolist()  # surface form of the word
    texts = rows["sentence"].tolist()

    collected = {layer_id: [] for layer_id in all_layer_ids}

    for start in range(0, len(rows), batch_size):
        end = min(start + batch_size, len(rows))

        batch_out = embedder.embed(
            words=words[start:end], # slice of target words
            target_texts=texts[start:end], # slice of sentences associated with target words
            layers_id=layers, # layers at which to store embeddings
            batch_size=batch_size,
            show_progress=False,
            averaging=True,
            return_static=True,
        )

        # store embeddings at each layer
        for layer_id in all_layer_ids:
            collected[layer_id].extend(batch_out[layer_id])

    records = []
    for i, row in rows.iterrows():
        records.append(
            {
                "word": row["word"], # kemma
                "string": row["string"], # actual form in sentence
                "meaning_id": row["meaning_id"], # e.g. M1_a
                "sentence": row["sentence"],
                "ambiguity_type": row["ambiguity_type"],  # hom/pol
                "embedding_static": collected[-1][i],
                "embedding_l4": collected[4][i],
                "embedding_l8": collected[8][i],
                "embedding_l12": collected[12][i],
            }
        )

    return records


# test
small_test = stimuli_long.head(8)
small_records = extract_embeddings(small_test, batch_size=4)

print(f"Small test completed: {len(small_records)} entries")
print("Example test entry keys:", list(small_records[0].keys()))
print(
    "Vector shapes (static, l4, l8, l12):",
    small_records[0]["embedding_static"].shape,
    small_records[0]["embedding_l4"].shape,
    small_records[0]["embedding_l8"].shape,
    small_records[0]["embedding_l12"].shape,
)

Final stimuli: 112
Sentence entries to embed (4 per stimulus): 448


Text tokenization: 100%|██████████| 4/4 [00:00<00:00, 972.65 examples/s]
d:\computational_linguistics_assignment2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 4/4 [00:00<00:00, 928.61 examples/s]
d:\computational_linguistics_assignment2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Small test completed: 8 entries
Example test entry keys: ['word', 'string', 'meaning_id', 'sentence', 'ambiguity_type', 'embedding_static', 'embedding_l4', 'embedding_l8', 'embedding_l12']
Vector shapes (static, l4, l8, l12): (768,) (768,) (768,) (768,)


In [ ]:
# Full

# extract embeddings for the whole dataset
embedding_records = extract_embeddings(stimuli_long, batch_size=32)
print(f"Full extraction completed: {len(embedding_records)} entries")

out_path = Path("raw-c/data/processed/rawc_contextual_embeddings.pkl")
out_path.parent.mkdir(parents=True, exist_ok=True)
with out_path.open("wb") as f:
    pickle.dump(embedding_records, f)

with out_path.open("rb") as f:
    loaded_records = pickle.load(f)

print(f"Loaded {len(loaded_records)} entries from {out_path}")

bank_records = [r for r in loaded_records if r["word"] == "bank"]
print(f"\nNumber of entries for word='bank': {len(bank_records)}")

# TEST FOR BANK
bank_df = pd.DataFrame(bank_records)
if not bank_df.empty:
    print(
        bank_df[
            [
                "word",
                "string",
                "meaning_id",
                "sentence",
                "embedding_static",
                "embedding_l4",
                "embedding_l8",
                "embedding_l12",
            ]
        ]
    )

Text tokenization: 100%|██████████| 32/32 [00:00<?, ? examples/s]
d:\computational_linguistics_assignment2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 32/32 [00:00<00:00, 3401.19 examples/s]
d:\computational_linguistics_assignment2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 32/32 [00:00<00:00, 4128.12 examples/s]
d:\computational_linguistics_assignment2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 32

Full extraction completed: 448 entries
Loaded 448 entries from raw-c\data\processed\rawc_contextual_embeddings.pkl

Number of entries for word='bank': 4
   word  string meaning_id                   sentence  \
0  bank  banked       M1_a       He banked the plane.   
1  bank  banked       M1_b  He banked the helicopter.   
2  bank  banked       M2_a       He banked the money.   
3  bank  banked       M2_b        He banked the cash.   

                                    embedding_static  \
0  [-0.032906216, -0.023735028, -0.04939177, -0.0...   
1  [-0.032906216, -0.023735028, -0.04939177, -0.0...   
2  [-0.032906216, -0.023735028, -0.04939177, -0.0...   
3  [-0.032906216, -0.023735028, -0.04939177, -0.0...   

                                        embedding_l4  \
0  [0.26263642, -0.19250788, -0.7618483, -0.00908...   
1  [0.23791067, -0.14989159, -0.6597786, -0.06863...   
2  [0.42974025, 0.27631962, -0.3599904, 0.0248075...   
3  [0.3850463, 0.28604165, -0.25840953, 0.0243157...   


# Task 2: Compute sense embeddings for the homonym dataset using WordNet [20 points]

Your next task is to fetch the definitions (glosses) of the homonyms, and compute an embedding for each gloss (each gloss is associated with a specific sense). We do that so we can later see whether the contextualised embeddings computed above represent the meaning of the homonym in context well (by comparing it to the sense embeddings). Figure 18.9 in [Jurafsky's and Martin's (2021) chapter 18](https://web.stanford.edu/~jurafsky/slp3/old_sep21/18.pdf) graphically illustrates this idea. Use this chapter for this part of the assignment, as it will come useful for you both theoretically and practically.

## Task 2.1: Fetch senses and glosses for a word [5 points]

First of all, you will have to figure out how [WordNet](https://www.nltk.org/howto/wordnet.html) works within the nltk package (hint: pay attention to what a synset is).

Install and import all the necessary components and define a function to extract the glosses of a word and create a dictionary with senses and glosses.

Then use the word "bat" to test that everything is working correctly: i.e., for "bat", you should be able to get its senses and the gloss for each of the sense (you will see that synsets might contain related words, but you only need the senses that contain the word of interest or derivates thereof; this should be specified in the function). Print the output for "bat".


In [29]:
import nltk

# download WordNet
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.corpus import wordnet as wn
from nltk.stem import PorterStemmer

ps = PorterStemmer()

# MAYBE TOO STRICT?  you only need the senses that contain the word of interest or DERIVATIVES, but we only keep the exact word of interest
def get_senses_and_glosses(word, pos=None):
    """
    Return synset_name -> gloss for strict exact matches of `word`.

    Rules:
    1) `word` must appear as an exact lemma in the synset.
    2) The synset canonical name must start with `word` (e.g., bat.n.01).
    """
    target = word.lower().strip() # clean the target word

    senses = {} # dictionary to be returned
    for syn in wn.synsets(target, pos=pos): # containing the target word
        lemma_names = {lemma.name().lower() for lemma in syn.lemmas()} # extract lemma associated with that meaning + clean them
        syn_head = syn.name().split(".", 1)[0].lower() # extract the dominant meaning in the synset

        # If the target word is found in the synset and is the leading meaning
        if target in lemma_names and syn_head == target: # MAYBE TOO STRICT
            senses[syn.name()] = syn.definition()  # retrieve its definitions and save in the dictionary of senses

    return senses

bat_senses = get_senses_and_glosses("bat")
print(f"Number of senses kept for 'bat': {len(bat_senses)}")
for sense, gloss in bat_senses.items():
    print(f"- {sense}: {gloss}")

Number of senses kept for 'bat': 7
- bat.n.01: nocturnal mouselike mammal with forelimbs modified to form membranous wings and anatomical adaptations for echolocation by which they navigate
- bat.n.02: (baseball) a turn trying to get a hit
- bat.n.05: a club used for hitting a ball in various games
- bat.v.01: strike with, or as if with a baseball bat
- bat.v.02: wink briefly
- bat.v.03: have a turn at bat
- bat.v.04: use a bat


In [30]:
wn.synsets('bat')

[Synset('bat.n.01'),
 Synset('bat.n.02'),
 Synset('squash_racket.n.01'),
 Synset('cricket_bat.n.01'),
 Synset('bat.n.05'),
 Synset('bat.v.01'),
 Synset('bat.v.02'),
 Synset('bat.v.03'),
 Synset('bat.v.04'),
 Synset('cream.v.02')]

## Task 2.2: Function to compute sense embeddings [10 points]

Now that you have a function to extract senses and glosses for a given word, write a function that takes a word and computes embeddings for each of the senses following the method explained in Jurafsky's and Martin's chapter. In this case, no need to calculate at different layers: you should use the last layer only. You should maximally optimise this function like before.

The output should include the sense, the gloss, and the embedding. Print the function's output when using the word "bank".


In [ ]:
def compute_sense_embeddings(word, batch_size=32):
    """
    Compute one embedding per WordNet sense gloss for `word`.
    Uses BERT last layer only (layer 12 for bert-base-uncased).

    Returns: list of dicts with keys: sense, gloss, embedding
    """

    # passes a target word as a parameter and retrieves the senses and glosses from WordNet with the function written above
    senses = get_senses_and_glosses(word)
    if not senses:
        return []

    # separate the senses and their definitions
    sense_ids = list(senses.keys())
    glosses = list(senses.values())

    target = word.lower().strip()

    # concatenate the target word with the definition with :, as a single list element
    gloss_contexts = [f"{target}: {gloss}" for gloss in glosses]
    words = [target] * len(gloss_contexts) # a list of just the target word, as long as the number of senses it has according to WordNet

    # compute embeddings for the senses
    layer_out = embedder.embed(
        words=words, # sanm, because we chose the strict approach and no derivatives
        target_texts=gloss_contexts, # definition for each sense
        layers_id=[12], # only at the last layer, as per the assignment
        batch_size=batch_size,
        show_progress=False,
        averaging=True,
        return_static=False, # no need
    )

    vectors = layer_out[12]

    # Create a list that holds all of the senses of a word, their definitions and contextualized vector embeddings (based on the definition from WordNet)
    records = []
    for i, sense_id in enumerate(sense_ids):
        records.append(
            {
                "sense": sense_id,
                "gloss": glosses[i],
                "embedding": vectors[i],
            }
        )

    return records


# test
bank_sense_embeddings = compute_sense_embeddings("bank", batch_size=32)
print(f"Number of sense embeddings for 'bank': {len(bank_sense_embeddings)}")

for rec in bank_sense_embeddings:
    print(f"\nSense: {rec['sense']}")
    print(f"Gloss: {rec['gloss']}")
    print(f"Embedding shape: {rec['embedding'].shape}")
    print(f"Embedding (first 10 values): {np.round(rec['embedding'][:10], 4)}")

print("\nFull output structure:")
print(bank_sense_embeddings)

Text tokenization: 100%|██████████| 14/14 [00:00<?, ? examples/s]
d:\computational_linguistics_assignment2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Number of sense embeddings for 'bank': 14

Sense: bank.n.01
Gloss: sloping land (especially the slope beside a body of water)
Embedding shape: (768,)
Embedding (first 10 values): [ 0.1215 -0.1408 -0.243  -0.2461  0.4397  0.4351 -0.5008  0.5747  0.4988
 -0.3803]

Sense: bank.n.03
Gloss: a long ridge or pile
Embedding shape: (768,)
Embedding (first 10 values): [ 0.0313 -0.224  -0.5094 -0.4397  0.576   0.2878 -0.3394  0.4803  0.3952
 -0.1734]

Sense: bank.n.04
Gloss: an arrangement of similar objects in a row or in tiers
Embedding shape: (768,)
Embedding (first 10 values): [ 0.3198 -0.2358 -0.7018  0.1425  0.3217  0.2789 -0.3238  0.1181  0.2064
  0.1765]

Sense: bank.n.05
Gloss: a supply or stock held in reserve for future use (especially in emergencies)
Embedding shape: (768,)
Embedding (first 10 values): [ 0.4284 -0.3351 -0.5739  0.0778  0.79    0.3652 -0.3313  0.5413  0.2452
  0.4157]

Sense: bank.n.06
Gloss: the funds held by a gambling house or the dealer in some gambling games
Embed

## Task 2.3: Compute sense embeddings for the RAW-C stimuli [5 points]

Now, use the function you defined above to compute sense embeddings for the RAW-C stimuli and pickle dump it too.

As above, the information that should be there for each word is: the sense, the gloss, the embedding at the last layer. Again, you can think of which structure to use best, but keep in mind that we will have to compare these to the CWE calculated in task 1, so it is good to think of a similar structure that is easily comparable.

Make sure that the number of stimuli matches the number of stimuli in the final RAW-C dataset.

In [32]:
from pathlib import Path
import pickle

# compute sense embeddings for each final RAW-C stimulus (word/string pair)
sense_embedding_records = []
for _, row in stimuli_final.iterrows():
    lemma = row["word"] # dict form
    lexeme = row["string"] # how it appears

    # extract the embeddings of the senses in the synset (from WordNet) for each word in the RAW-C dataset
    senses_for_word = compute_sense_embeddings(lemma, batch_size=32)
    # print(senses_for_word)
    sense_embedding_records.append(
        {
            "word": lemma,
            "string": lexeme,
            "sense_embeddings": senses_for_word,  # each item: sense, gloss, embedding
        }
    )

print(f"Computed sense embeddings for stimuli: {len(sense_embedding_records)}")
print(f"Expected final RAW-C stimuli count: {len(stimuli_final)}")

# pickle dumpp
out_path = Path("raw-c/data/processed/rawc_sense_embeddings.pkl")
out_path.parent.mkdir(parents=True, exist_ok=True)
with out_path.open("wb") as f:
    pickle.dump(sense_embedding_records, f)

# load back to test
with out_path.open("rb") as f:
    loaded_sense_records = pickle.load(f)

print(f"Loaded {len(loaded_sense_records)} entries from {out_path}")

# test
bank_entries = [r for r in loaded_sense_records if r["word"] == "bank"]
print(f"\nNumber of entries for word='bank': {len(bank_entries)}")
if bank_entries:
    bank_entry = bank_entries[0]
    print("word:", bank_entry["word"])
    print("string:", bank_entry["string"])
    print("number of senses:", len(bank_entry["sense_embeddings"]))

    for s in bank_entry["sense_embeddings"][:3]:
        print("- sense:", s["sense"])
        print("  gloss:", s["gloss"])
        print("  embedding shape:", s["embedding"].shape)

assert len(loaded_sense_records) == len(stimuli_final), "Stimulus count mismatch."

Text tokenization: 100%|██████████| 13/13 [00:00<?, ? examples/s]
d:\computational_linguistics_assignment2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 5/5 [00:00<00:00, 685.97 examples/s]
d:\computational_linguistics_assignment2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 4/4 [00:00<00:00, 830.97 examples/s]
d:\computational_linguistics_assignment2\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Text tokenization: 100%|██████████| 7/7 [00:

Computed sense embeddings for stimuli: 112
Expected final RAW-C stimuli count: 112
Loaded 112 entries from raw-c\data\processed\rawc_sense_embeddings.pkl

Number of entries for word='bank': 1
word: bank
string: banked
number of senses: 14
- sense: bank.n.01
  gloss: sloping land (especially the slope beside a body of water)
  embedding shape: (768,)
- sense: bank.n.03
  gloss: a long ridge or pile
  embedding shape: (768,)
- sense: bank.n.04
  gloss: an arrangement of similar objects in a row or in tiers
  embedding shape: (768,)


# Task 3: Compute and explore similarity between homonym CWEs and sense embeddings [35 points]

You now have the homonym CWEs computed in task 1, and the sense embeddings computed in task 2. The next step is to calculate cosine similarities between each CWE for each homonym (at the selected layer!) and each sense embedding for that homonym.

For instance, say for the word "bat" with meaning M1_a, you have its CWE at the static layer and at layers 4, 8, 12 and 7 senses: here, you will end up with 16 cosine similarities (take each CWE and compute its similarity to each of the sense embeddings). We then want to see which sense meaning is the closest to each CWE, and do some qualitative explorations with that.

## Task 3.1: Compute the cosine similarity between all the CWEs and the sense embeddings [8 points]

This task is not trivial with regards to how much information you have and how to structure the data (this is why it's also important to think of data structures in the earlier parts of the assignment), so take some time to think how to best breakdown this task. Test each step/function if you have multiple. Pickle dump your final output so that it is easily retrievable for later. At the end, print an example of the entry "bank".

For cosine similarity, the cdist function from scipy.spatial.distance seems the most efficient, but you are free to use any of your liking (hint: pay attention to the shape of your embeddings and to similarity vs distance. You will need the similarity).

In [ ]:
#your code here
from pathlib import Path
import pickle

# Load sense embeddings from WordNet
sense_embeddings = Path("raw-c/data/processed/rawc_sense_embeddings.pkl")

with sense_embeddings.open("rb") as f:
    loaded_sense_records = pickle.load(f)

# Load CWEs from RAW-C
context_embeddings = Path("raw-c/data/processed/rawc_contextual_embeddings.pkl")

with context_embeddings.open("rb") as f:
    loaded_contextual_records = pickle.load(f)



In [33]:
from sklearn.metrics.pairwise import cosine_similarity

# MAYBE ALSO USE STRING (DIFFERENT FORM, SAME MEANING)
# transform list sense embeddings to dict to match the format
def transform_sense_embeddings(listA: list) -> dict:
    return {obj['word']: obj for obj in listA}

loaded_sense_records_dict = transform_sense_embeddings(loaded_sense_records)

computed_similarities = {} # stored in a dict
for obj in loaded_contextual_records: # for embedding of each target word in RAW-C
    word = obj['word'] 
    meaning_id = obj['meaning_id']
    sense_embeddings_list = loaded_sense_records_dict[word]['sense_embeddings'] # retrieve ALL of the sense embeddings for that word (computed from WordNet)

    sense_sims = {}
    for sense in sense_embeddings_list: # for each sense of a word in RAW-C
        sense_vec = sense['embedding'].reshape(1, -1) # retrieve its embedding and reshape for cosine similarity

        # compute similarity at each layer
        # Use [0][0] because cosine_similarity returns a [1][1] matrix, so we want to extract the value itself
        sense_sims[sense['sense']] = {
            'static': cosine_similarity(obj['embedding_static'].reshape(1, -1), sense_vec)[0][0],
            'L4':     cosine_similarity(obj['embedding_l4'].reshape(1, -1),     sense_vec)[0][0],
            'L8':     cosine_similarity(obj['embedding_l8'].reshape(1, -1),     sense_vec)[0][0],
            'L12':    cosine_similarity(obj['embedding_l12'].reshape(1, -1),    sense_vec)[0][0],
        }

    computed_similarities[(word, meaning_id)] = sense_sims

In [34]:
computed_similarities

{('act',
  'M1_a'): {'act.n.01': {'static': np.float32(0.1424515),
   'L4': np.float32(0.285096),
   'L8': np.float32(0.30678365),
   'L12': np.float32(0.29337582)}, 'act.n.02': {'static': np.float32(0.15137306),
   'L4': np.float32(0.26215637),
   'L8': np.float32(0.2839592),
   'L12': np.float32(0.27838004)}, 'act.n.03': {'static': np.float32(0.22463521),
   'L4': np.float32(0.30715024),
   'L8': np.float32(0.31913608),
   'L12': np.float32(0.36363247)}, 'act.n.04': {'static': np.float32(0.17319953),
   'L4': np.float32(0.26937705),
   'L8': np.float32(0.26643965),
   'L12': np.float32(0.23531364)}, 'act.n.05': {'static': np.float32(0.13590139),
   'L4': np.float32(0.27556396),
   'L8': np.float32(0.30214965),
   'L12': np.float32(0.26221836)}, 'act.v.01': {'static': np.float32(0.12258984),
   'L4': np.float32(0.2369961),
   'L8': np.float32(0.27211773),
   'L12': np.float32(0.27992105)}, 'act.v.02': {'static': np.float32(0.105589315),
   'L4': np.float32(0.24930134),
   'L8': np.flo

## Task 3.2: Quantitative and qualitative explorations the relationship between homonym embeddings and dominant senses

Now, we can look into how the CWEs in different meanings and layers relate to the different senses of a homonym. We'll focus on the dominant sense in WordNet, see below for more details. This section includes both code blocks and reflection questions.

### Dominant senses in WordNet and top senses across layers (focus on static layer) [8 points]

Embeddings at the static layer do not take into account context, so intuitively they should capture the 'average' meaning, maybe the most common/dominant. We can test this by looking at the most similar sense and seeing if that matches that most common/dominant sense in the synset.

Keep in mind that synsets mark more common/dominant senses with numbering: so n.01 will be the most common noun; v.01 the most common verb, etc. If that is not available, the most common meaning will be the next number (e.g., n.02). You have to take that into account when you extract the top sense, so first extract information about which are the most dominant senses for each word across all the parts of speech: for example, "bat" might have as its two most common senses bat.n.01 and bat.v.02 (because v.01 might not be available; this is just an example). Some words might only have one part of speech in their synset, some more. Print your results.

In [17]:
from collections import Counter


In [18]:
homonymy = rawc[rawc['ambiguity_type'] == 'Homonymy']
homonymy_set = set(homonymy['word'].unique().tolist())

# # Check if all homonyms are homonyms
# for homonym_w in homonymy_set:
#     print(rawc[rawc['word'] == homonym_w ]['ambiguity_type'])

In [37]:
from collections import Counter

similarity_layers = ['static','L4','L8','L12']

cnt_static_layer = Counter()
prev_pair = None

for (word, meaning_id), senses_similarities in computed_similarities.items():

    pairs = {}

    # build pairs using STATIC layer explicitly
    for sense, values in senses_similarities.items():
        sim = values['static']   # explicitly use static layer
        pairs[sim] = (sense, word)

    # highest similarity
    highest = max(pairs.keys())

    # --- NEW: dominance check (minimal addition) ---
    best_sense = pairs[highest][0]
    lemma, pos, rank = best_sense.split('.')
    rank = int(rank)

    synsets = wn.synsets(word)
    has_pos_01 = any(s.name().startswith(f"{lemma}.{pos}.01") for s in synsets)

    if has_pos_01:
        is_dominant = (rank == 1)
    else:
        ranks_for_pos = [
            int(s.name().split('.')[-1])
            for s in synsets
            if s.name().split('.')[1] == pos
        ]
        min_rank = min(ranks_for_pos) if ranks_for_pos else None
        is_dominant = (rank == min_rank)
    # --- END NEW ---

    if pairs[highest] == prev_pair:
        continue
    else:
        print(pairs[highest], highest, "| dominant:", is_dominant)

    # extract POS + rank (e.g., n.01)
    pos_rank = f"{pairs[highest][0].split('.')[-2]}.{pairs[highest][0].split('.')[-1]}"
    cnt_static_layer[pos_rank] += 1

    prev_pair = pairs[highest]

('act.n.03', 'act') 0.22463521 | dominant: False
('appeal.n.02', 'appeal') 0.19151682 | dominant: False
('atmosphere.n.03', 'atmosphere') 0.2747226 | dominant: False
('bail.v.03', 'bail') 0.19301623 | dominant: False
('band.v.01', 'band') 0.23327664 | dominant: True
('bank.v.07', 'bank') 0.174775 | dominant: False
('barrier.n.02', 'barrier') 0.23496339 | dominant: False
('bat.n.05', 'bat') 0.21835303 | dominant: False
('beam.v.03', 'beam') 0.23708782 | dominant: False
('block.v.06', 'block') 0.25064743 | dominant: False
('blood.n.02', 'blood') 0.24408045 | dominant: False
('board.n.04', 'board') 0.1937581 | dominant: False
('book.n.02', 'book') 0.24364819 | dominant: False
('break.n.12', 'break') 0.21229082 | dominant: False
('breakfast.n.01', 'breakfast') 0.24724707 | dominant: True
('call.v.19', 'call') 0.14227131 | dominant: False
('cape.n.01', 'cape') 0.25832295 | dominant: True
('case.n.19', 'case') 0.24010783 | dominant: False
('cast.n.07', 'cast') 0.2499541 | dominant: False
('c

In [38]:
print("Most prevelant types of senses")
print(cnt_static_layer)

Most prevelant types of senses
Counter({'n.01': 20, 'n.02': 17, 'n.03': 12, 'n.05': 11, 'n.04': 8, 'v.01': 5, 'n.08': 5, 'v.03': 4, 'n.07': 4, 'n.06': 4, 's.01': 3, 'v.02': 3, 'v.04': 3, 'v.06': 2, 'v.05': 2, 'v.07': 1, 'n.12': 1, 'v.19': 1, 'n.19': 1, 'v.23': 1, 'r.01': 1, 'v.10': 1, 'n.13': 1, 'n.10': 1})


Then, extract the top similarity of homonyms to the senses at all the layers you have available. While we are interested in the static layer for checking dominant senses, it is also interesting to look into other layers to see whether adding context will refine the captured meaning.


In [39]:
cnt_all_layers = Counter()

bank_list = []
for word,meaning_id in computed_similarities:
    # if word not in homonymy_set:
        # continue
    senses_similarities = computed_similarities[word,meaning_id]
    pairs = {}
    for sense,value in senses_similarities.items():
        list_similarities = [v for v in value.values()]
        max_similarity = max(list_similarities)
        max_similarity_idx = list_similarities.index(max_similarity)
        pairs[max_similarity] = (sense,word,meaning_id,similarity_layers[max_similarity_idx])
    # higher cosine similarity, more similar words
    highest = max([k for k in pairs.keys()])
    
    if word == 'bank':
        bank_list.append((pairs[highest],highest))
    print(pairs[highest],highest)
    cnt_all_layers[f"{pairs[highest][-1]}"] += 1

('act.n.03', 'act', 'M1_a', 'L12') 0.36363247
('act.n.03', 'act', 'M1_b', 'L12') 0.40399146
('act.n.03', 'act', 'M2_a', 'L12') 0.4674677
('act.n.03', 'act', 'M2_b', 'L12') 0.51158255
('appeal.n.02', 'appeal', 'M1_a', 'L12') 0.37684214
('appeal.n.02', 'appeal', 'M1_b', 'L12') 0.43116012
('appeal.n.03', 'appeal', 'M2_a', 'L12') 0.45925963
('appeal.n.03', 'appeal', 'M2_b', 'L12') 0.4616029
('atmosphere.n.03', 'atmosphere', 'M1_a', 'L4') 0.40705007
('atmosphere.n.03', 'atmosphere', 'M1_b', 'L12') 0.44561285
('atmosphere.n.03', 'atmosphere', 'M2_a', 'L12') 0.62226665
('atmosphere.n.03', 'atmosphere', 'M2_b', 'L12') 0.49775487
('bail.v.04', 'bail', 'M1_a', 'L8') 0.4224473
('bail.v.04', 'bail', 'M1_b', 'L8') 0.41571066
('bail.v.05', 'bail', 'M2_a', 'L8') 0.43369538
('bail.v.05', 'bail', 'M2_b', 'L8') 0.40900043
('band.n.04', 'band', 'M1_a', 'L12') 0.57044697
('band.n.13', 'band', 'M1_b', 'L12') 0.5573489
('band.n.02', 'band', 'M2_a', 'L12') 0.59575534
('band.n.02', 'band', 'M2_b', 'L12') 0.60

In [40]:
print("Most prevelant layers")
cnt_all_layers

Most prevelant layers


Counter({'L12': 296, 'L8': 110, 'L4': 42})

Let's check an example from our results.

Out of all the similarities of 'bank' to all its senses at all the layers, which one is the highest? Print your results for that entry and reflect below.

In [41]:
bank_list

[(('bank.n.10', 'bank', 'M1_a', 'L8'), np.float32(0.35752672)),
 (('bank.n.10', 'bank', 'M1_b', 'L8'), np.float32(0.36990333)),
 (('bank.n.06', 'bank', 'M2_a', 'L12'), np.float32(0.42032656)),
 (('bank.n.06', 'bank', 'M2_b', 'L8'), np.float32(0.39070493))]

### Does the static layer capture the most dominant meaning, according to WordNet (and according to you)? [2 point]

The static layer does not reliably capture the most dominant meaning according to WordNet. Although the most frequent sense (n.01) is selected more often than any individual alternative, the combined frequency of lower-ranked senses (e.g., n.02, n.03, etc.) is substantially higher. This indicates that static embeddings do not consistently align with the dominant sense, but instead reflect a blended representation of multiple meanings. This is expected, as static embeddings do not incorporate contextual information and therefore cannot disambiguate between different senses of a word.

### Across other layers and meanings, which layer seems to capture the meaning of bank across meanings best, and why do you make this conclusion? [2 points]

For the word "bank", deeper layers (L8 and L12) appear to capture meaning best. While layer 12 produces the highest cosine similarity score, layer 8 is more consistently selected across different meanings. Importantly, both layers successfully distinguish between the two meanings of "bank", assigning one sense to M1 contexts and another to M2 contexts. This suggests that contextual information in deeper layers allows the model to refine word meaning representations and perform effective sense disambiguation.


### Checking matches and mismatches with the dominant sense [5 points]

Now, let's quantitatively check if the static layer actually captures the most dominant sense (any POS). You should end up with two data structures: matches (when the most similar sense is one of the dominant senses) and mismatches (when the most similar sense is not one of the dominant sense). Do that also for the other layers to compare. Print the percentage of matches and mismatches per layer.



In [42]:
similarity_layers = ['static','L4','L8','L12']

# initialize lists
matches = {layer: [] for layer in similarity_layers} 
mismatches = {layer: [] for layer in similarity_layers}

for (word, meaning_id), senses_similarities in computed_similarities.items():

    synsets = wn.synsets(word)

    for layer in similarity_layers:

        pairs = {}

        # build pairs exactly like before, but per layer
        for sense, value in senses_similarities.items():
            sim = value[layer]
            pairs[sim] = sense

        highest = max(pairs.keys())
        best_sense = pairs[highest]

        # --- dominant check (same logic as before) ---
        lemma, pos, rank = best_sense.split('.')
        rank = int(rank)

        has_pos_01 = any(s.name().startswith(f"{lemma}.{pos}.01") for s in synsets)

        if has_pos_01:
            is_dominant = (rank == 1)
        else:
            ranks_for_pos = [
                int(s.name().split('.')[-1])
                for s in synsets
                if s.name().split('.')[1] == pos
            ]
            min_rank = min(ranks_for_pos) if ranks_for_pos else None
            is_dominant = (rank == min_rank)
        # --- end check ---

        if is_dominant:
            matches[layer].append((word, meaning_id, best_sense))
        else:
            mismatches[layer].append((word, meaning_id, best_sense))

Now, print the matches and mismatches for the static layer only.

In [ ]:
for layer in similarity_layers:
    total = len(matches[layer]) + len(mismatches[layer])
    match_pct = len(matches[layer]) / total * 100
    mismatch_pct = len(mismatches[layer]) / total * 100

    print(f"\nLayer: {layer}")
    print(f"Matches: {len(matches[layer])} ({match_pct:.2f}%)")
    print(f"Mismatches: {len(mismatches[layer])} ({mismatch_pct:.2f}%)")


Layer: static
Matches: 116 (25.89%)
Mismatches: 332 (74.11%)

Layer: L4
Matches: 87 (19.42%)
Mismatches: 361 (80.58%)

Layer: L8
Matches: 88 (19.64%)
Mismatches: 360 (80.36%)

Layer: L12
Matches: 110 (24.55%)
Mismatches: 338 (75.45%)


In [46]:
static_mismatches = mismatches['static']

words_static_mismatch = [w for (w, _, _) in static_mismatches]

word_counts = Counter(words_static_mismatch)
print(word_counts.most_common(100))

static_matches = matches['static']

words_static_match = [w for (w, _, _) in static_matches]

word_counts_matches = Counter(words_static_match)
print(word_counts_matches.most_common(100))

[('act', 4), ('appeal', 4), ('atmosphere', 4), ('bail', 4), ('bank', 4), ('barrier', 4), ('bat', 4), ('beam', 4), ('block', 4), ('blood', 4), ('board', 4), ('book', 4), ('break', 4), ('call', 4), ('case', 4), ('cast', 4), ('cause', 4), ('cell', 4), ('charm', 4), ('check', 4), ('clip', 4), ('clog', 4), ('column', 4), ('company', 4), ('cone', 4), ('contribution', 4), ('cross', 4), ('date', 4), ('degree', 4), ('design', 4), ('draw', 4), ('drill', 4), ('examination', 4), ('fan', 4), ('film', 4), ('fix', 4), ('ground', 4), ('guard', 4), ('guide', 4), ('hail', 4), ('home', 4), ('intelligence', 4), ('issue', 4), ('jam', 4), ('lamb', 4), ('lap', 4), ('load', 4), ('market', 4), ('match', 4), ('medicine', 4), ('mold', 4), ('mole', 4), ('movement', 4), ('newspaper', 4), ('orange', 4), ('page', 4), ('palm', 4), ('panel', 4), ('passage', 4), ('perch', 4), ('pick', 4), ('pitcher', 4), ('port', 4), ('position', 4), ('post', 4), ('punch', 4), ('pupil', 4), ('racket', 4), ('rock', 4), ('run', 4), ('sca

### Do BERT's static embeddings capture the most dominant sense in WordNet? [2 point]

No layer appears to capture the most dominant WordNet sense reliably. Match rates remain low across all layers (around 20–26%). Interestingly, the static layer and the 12th layer perform slightly better than intermediate layers (L4 and L8), but still only identify the dominant sense in approximately 25% of cases.

### Do the percentages of matches and mismatches throughout the layers make sense to you or is it different than what you expected? [2 points]

It is reasonable that the static layer performs relatively well, as it captures an average, context-independent representation of the word. However, it is unexpected that deeper layers, particularly L12, do not significantly improve alignment with dominant WordNet senses, despite being more effective at contextual disambiguation in earlier analyses. Additionally, the lower performance of intermediate layers (L4 and L8) suggests that these layers may capture transitional representations that are neither purely general nor fully contextualized.

### For the **static layer**, are there any words that seem to particularly deviate from the dominant meaning? If so, which and why could that be? [3 points]

Firstly, we found that the static layer matches the dominant WordNet sense in only about 25% of cases, indicating that it struggles to consistently capture the dominant meaning. However, a clear pattern emerges when comparing matches and mismatches. 

Words that are correctly matched at the static layer tend to have a clear, dominant meaning and relatively low ambiguity. These words are often used primarily in a single part of speech and have fewer competing senses. For example, the word “breakfast” is typically used as a noun with a well-defined meaning, which allows the static embedding to align more closely with the dominant sense. 

In contrast, mismatches tend to involve highly ambiguous words with multiple meanings and often multiple parts of speech. For example, the word “act” can function as both a noun (e.g., an act in a play) and a verb (e.g., to act in a certain way). Because these words appear in a wide variety of contexts, their static embeddings represent a mixture of meanings rather than a single dominant sense. As a result, they are more likely to deviate from the dominant WordNet sense. 

Overall, the degree of ambiguity and variability in usage appears to be a key factor influencing whether a word deviates from its dominant meaning at the static layer. 

### Do you think the corpus on which BERT is trained might reflect different meaning dominance than for WordNet's senses? If so/not, why? [3 points]

Although the static layer is expected to capture an “average” meaning of a word, the results show that this is not the case in practice. The static layer selects the dominant WordNet sense only about 25% of the time, indicating that it does not reliably capture the most common meaning. The discrepancy may reflect differences between the distribution of meanings in the corpus on which BERT was trained and the sense hierarchy defined in WordNet.

# Task 4: Partially replicate Trott & Bergen's experiment [20 points]

Now comes the time to partially replicate the RAW-C experiment, by seeing whether different layers of BERT capture meanings more or less similarly to humans. At the end you will have to wrap up with a brief comment on which layer seems to capture meanings best and how that connects to explorations in the previous section.

## Task 4.1: Create a dataframe with cosine similarities between sentences at different layers [7 points]

You should now use the embeddings at the different layers that you computed to calculate similarities between each context: M1a, M1b, M2a, M2b. You will have to have all combinations, so for each string in the RAW-C dataframe, you'll have: M1a vs M1b, M1a vs M2a, M1a vs M2b, M1b vs M2a, M1b vs M2b, M2a vs M2b.

Bear in mind that your final dataframe should include: the word, the string as it appears in the sentence, cosine similarity at layers 4, layer 8, layer 12, the version being compared (is it M1a vs M1b or M1a vs M2a?) and the mean relatadness given by humans (hint: the repo you cloned will come useful here, both in terms of code and data). Print the head of the dataframe to check everything is in order, and check also that the number of stimuli match with your number across the assignment (starting from task 1).

In [ ]:
#your code here

## Task 4.2: Correlate with human judgements and visualise [8 points]

First, correlate the cosine similarities at the different layers to the mean human relatedness judgements. Use the same correlation metric used by Trott & Bergen.

In [ ]:
#your code here

Next, visualise your results. You want to see the correlation between BERT embeddings and human judgements per layer, but what would also be interesting is to include the meaning contrasts (such as M1_a_M1_b, etc), so that we can see how those play out per layer.

In [ ]:
#your code here

### Reflect on the correlations and on the visualisations. What can you observe and infer in terms of which layer(s) might be capturing meaning best? Is there one way to determine that (i.e., what does 'capturing meanings' mean?)? Contrast and compare the layers. [5 points]

%your answer here



